### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
# Dependency bootstrap (auto-install missing packages)
import importlib.util
import subprocess
import sys

PKG_MAP = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}

missing = [pkg for mod, pkg in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing missing dependencies: {missing}')
    if sys.platform == 'emscripten':
        import micropip
        await micropip.install(missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready.')


# Kernel Lift for Nonlinear Separation

This notebook shows the same data in two views: the original 2D space where the classes are not linearly separable, and a lifted 3D feature space using $z = x_1^2 + x_2^2$ where a horizontal plane can separate them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from mpl_toolkits.mplot3d import Axes3D


In [ ]:
theta_inner = np.linspace(0, 2 * np.pi, 18, endpoint=False)
theta_outer = np.linspace(0, 2 * np.pi, 28, endpoint=False)
INNER = np.column_stack([0.75 * np.cos(theta_inner), 0.75 * np.sin(theta_inner)])
OUTER = np.column_stack([1.9 * np.cos(theta_outer), 1.9 * np.sin(theta_outer)])
INNER_Z = np.sum(INNER ** 2, axis=1)
OUTER_Z = np.sum(OUTER ** 2, axis=1)

def plot_kernel_lift(radius_threshold=1.35):
    plane_z = radius_threshold ** 2
    fig = plt.figure(figsize=(11.5, 5.4))
    ax1 = fig.add_subplot(1, 2, 1)
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')

    ax1.scatter(INNER[:, 0], INNER[:, 1], c='#2563eb', s=42, label='Inner ring')
    ax1.scatter(OUTER[:, 0], OUTER[:, 1], c='#dc2626', s=42, label='Outer ring')
    circle = plt.Circle((0, 0), radius_threshold, fill=False, color='#0f766e', linewidth=2)
    ax1.add_patch(circle)
    ax1.text(-2.5, 2.25, 'Nonlinear boundary in the original space', fontsize=11, color='#0f172a')
    ax1.set_xlim(-2.6, 2.6)
    ax1.set_ylim(-2.6, 2.6)
    ax1.set_aspect('equal')
    ax1.grid(alpha=0.2)
    ax1.legend(loc='lower right')
    ax1.set_title('Original 2D Input Space')

    ax2.scatter(INNER[:, 0], INNER[:, 1], INNER_Z, c='#2563eb', s=34)
    ax2.scatter(OUTER[:, 0], OUTER[:, 1], OUTER_Z, c='#dc2626', s=34)
    xx, yy = np.meshgrid(np.linspace(-2.4, 2.4, 14), np.linspace(-2.4, 2.4, 14))
    zz = np.full_like(xx, plane_z)
    ax2.plot_surface(xx, yy, zz, alpha=0.25, color='#0f766e', linewidth=0)
    ax2.set_xlim(-2.6, 2.6)
    ax2.set_ylim(-2.6, 2.6)
    ax2.set_zlim(0, 5)
    ax2.set_xlabel('x1')
    ax2.set_ylabel('x2')
    ax2.set_zlabel('x1^2 + x2^2')
    ax2.set_title('Lifted Feature Space')
    ax2.view_init(elev=23, azim=-58)
    plt.tight_layout()
    plt.show()
    print(f'Separating plane: z = {plane_z:.2f}. Inner points lie mostly below it, outer points above it.')

widgets.interact(
    plot_kernel_lift,
    radius_threshold=widgets.FloatSlider(value=1.35, min=0.9, max=2.1, step=0.05, description='radius')
);
